<a href="https://colab.research.google.com/github/Magdyibrahim112/Evently_cycle16/blob/development/neural_project_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import requests

# =========================
# 1) Load WikiText-2 sample
# =========================
url = "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt"
raw_data = requests.get(url).text.split('\n')[:500]#بياخد أول 500 سطر بس (عشان السرعة)

# =========================
# 2) Tokenization
# =========================
def tokenize(sentence):#يحول الجملة لـ lowercase
    tokens = sentence.lower().strip().split()
    if len(tokens) < 3:
        return []
    return ["<s>"] + tokens + ["</s>"]
#يقسمها كلمات
#يضيف: <s> = بداية الجملة
#</s> = نهاية الجملة

tokenized_corpus = [tokenize(s) for s in raw_data if len(s) > 0]
#بيلف على كل سطر (جملة) في الداتا
#if len(s) > 0 بيشيل الجمل الفاضية

# =========================
# 3) Build Vocabulary
# =========================
vocab = set()#الـ set مفيهوش ترتيب ثابت

for sent in tokenized_corpus:
    vocab.update(sent)
#بيجمع كل الكلمات المختلفة في الداتا

vocab = list(vocab)#بقى list كان set

word_to_ix = {w: i for i, w in enumerate(vocab)}#يدي لكل كلمة رقم (ID)
ix_to_word = {i: w for w, i in word_to_ix.items()}# ده العكس تمامًا
#لما الموديل يتوقع رقم
#نحوله لكلمة مفهومة
vocab_size = len(vocab)# ده عدد الكلمات

print("Vocab size:", vocab_size)

# =========================
# 4) Build Trigrams Dataset
# =========================
#الجزء ده بيحوّل الجمل إلى Dataset جاهز للتدريب باستخدام
#فكرة الـ Trigram (يعني نتوقع كلمة من كلمتين قبلها)
trigrams = []

for sent in tokenized_corpus:
    for i in range(len(sent) - 2):#إحنا بناخد 3 كلمات كل مرة
        context = (sent[i], sent[i+1])#ده أول كلمتين
        target = sent[i+2]#دي الكلمة اللي عايزين الموديل يتوقعها
        trigrams.append((context, target))#بيضيف كل مثال للـ dataset

print("Training samples:", len(trigrams))

# =========================
# 5) Neural Language Model
# =========================
class NeuralLM(nn.Module):#يعني بنعمل موديل جديد باستخدام PyTorch
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)#vector بدل ما الكلمة تكون رقم بتتحول لـ
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        #📌 ليه * 2 ؟
        #لأن عندنا:
        #كلمتين (w1, w2)
        #كل واحدة لها embedding
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
        #📌 دي بتطلع:
        #Score لكل كلمة في الـ vocab

    def forward(self, w1, w2):#دي رحلة البيانات جوه الموديل
        e1 = self.embedding(w1)
        e2 = self.embedding(w2)
        x = torch.cat((e1, e2), dim=1)#دمجنا المعنى بتاع الكلمتين
        h = torch.tanh(self.fc1(x))
        out = self.fc2(h)
        return out

# =========================
# 6) Model Setup
# =========================
EMBED_DIM = 20#ده طول الـ vector اللي بيمثل كل كلمة
HIDDEN_DIM = 64#ده عدد النيورونات في الطبقة المخفية

model = NeuralLM(vocab_size, EMBED_DIM, HIDDEN_DIM)
#هنا بنبني الموديل باستخدام:
#عدد الكلمات (vocab_size)
#حجم الـ embedding
#حجم الطبقة المخفية


loss_fn = nn.CrossEntropyLoss()#تقارن بين توقع الموديل الإجابة الصح
optimizer = optim.Adam(model.parameters(), lr=0.005)
#تحديث أوزان الموديل
#🔸 Adam Optimizer : سريع وفعال , بيعدل الـ learning rate تلقائيًا

# =========================
# 7) Training Loop
# =========================هنا الموديل “بيتعلّم”
epochs = 20

for epoch in range(epochs):
    total_loss = 0
    #📌 في بداية كل epoch:
    #بنصفر الـ loss عشان نحسب متوسط الخطأ

    for i in range(min(len(trigrams), 2000)):
        (w1, w2), target = trigrams[i]

        if w1 not in word_to_ix or w2 not in word_to_ix or target not in word_to_ix:
            continue

        w1 = torch.tensor([word_to_ix[w1]])
        w2 = torch.tensor([word_to_ix[w2]])
        target = torch.tensor([word_to_ix[target]])

        output = model(w1, w2)
        loss = loss_fn(output, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/2000:.4f}")

# =========================
# 8) Prediction Function
# =========================
def predict_next_word(word1, word2):
    if word1 not in word_to_ix or word2 not in word_to_ix:
        return "Unknown word"

    model.eval()
    with torch.no_grad():
        w1 = torch.tensor([word_to_ix[word1]])
        w2 = torch.tensor([word_to_ix[word2]])

        output = model(w1, w2)
        predicted_index = torch.argmax(output).item()

    return ix_to_word[predicted_index]

Vocab size: 3802
Training samples: 22402
Epoch 1/20, Loss: 6.9683
Epoch 2/20, Loss: 4.4432
Epoch 3/20, Loss: 3.1852
Epoch 4/20, Loss: 2.4072
Epoch 5/20, Loss: 1.9800
Epoch 6/20, Loss: 1.7421
Epoch 7/20, Loss: 1.6324
Epoch 8/20, Loss: 1.5643
Epoch 9/20, Loss: 1.5652
Epoch 10/20, Loss: 1.5498
Epoch 11/20, Loss: 1.5002
Epoch 12/20, Loss: 1.5190
Epoch 13/20, Loss: 1.5044
Epoch 14/20, Loss: 1.5254
Epoch 15/20, Loss: 1.5062
Epoch 16/20, Loss: 1.4922
Epoch 17/20, Loss: 1.4913
Epoch 18/20, Loss: 1.4879
Epoch 19/20, Loss: 1.4936
Epoch 20/20, Loss: 1.5008


In [2]:
# Interactive testing for our simple language model

def generate_interactive():
    print("\n--- Simple Model Testing ---")
    print("This model predicts the 3rd word based on 2 words (Trigram idea)")
    print("Type 'exit' to stop\n")

    while True:
        word1 = input("First word: ").lower().strip()
        if word1 == "exit":
            break

        word2 = input("Second word: ").lower().strip()
        if word2 == "exit":
            break

        # check if words exist in vocabulary
        if word1 not in word_to_ix or word2 not in word_to_ix:
            print("Word not found in training data.\n")
            continue

        # get prediction
        result = predict_next_word(word1, word2)

        print("Predicted word:", result, "\n")


# run the test
generate_interactive()


--- Simple Model Testing ---
This model predicts the 3rd word based on 2 words (Trigram idea)
Type 'exit' to stop

First word: exit
